# Explanation:

1. BioBERT Model Training on BioASQ Dataset
2. DistilBERT on Squad Dataset for semantic Understanding
3. PDF File Processing
4. ChatBot Demo

# Library Install

In [ ]:
!pip install transformers
!pip install torch
!pip install -U sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 757.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.8 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=cc750267084a181f0fa4429c81b004fd34ada0d76c303f8f56ae42f43c2ed6e1
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


# Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load BioASQ dataset

In [ ]:
import json

file_path = '/content/drive/My Drive/ECE_7500_NLP/BioASQ/BioASQ-trainingDataset11b.json'

def load_bioasq_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data['questions']  # Assuming the key is 'questions'

bioasq_data = load_bioasq_dataset(file_path)
print(len(bioasq_data)) #4719

4719


# 1. BioBERT Model on the BioASQ Dataset



##Pre-Process

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertForQuestionAnswering

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def preprocess_data(data):
    # Initialize an empty list to hold processed data
    processed_data = []

    # Iterate over each item in the input data
    for item in data:
        # Extract the question text from the item
        question_text = item['body']

        # Iterate over each snippet in the item, if any
        for snippet in item.get('snippets', []):
            # Extract the context text from the snippet
            context_text = snippet['text']

            # Check if the item type is 'factoid' and it has an exact answer
            if item['type'] == 'factoid' and item.get('exact_answer'):
                # Extract the first exact answer
                answer = item['exact_answer'][0]

                # Encode the question and context texts using a tokenizer
                encoded_input = tokenizer.encode_plus(
                    question_text,
                    context_text,
                    add_special_tokens=True,  # Add special tokens (e.g., [CLS], [SEP])
                    max_length=512,          # Set maximum length for the encoded input
                    padding='max_length',    # Pad or truncate to a fixed length
                    truncation=True,         # Enable truncation to max_length
                    return_tensors='pt'      # Return PyTorch tensors
                )

                # Extract input IDs from the encoded input
                input_ids = encoded_input['input_ids'][0]

                # Tokenize the answer
                answer_tokens = tokenizer.tokenize(answer)

                # Try to find the start and end indices of the answer in the input IDs
                try:
                    start_token_idx = input_ids.tolist().index(tokenizer.convert_tokens_to_ids(answer_tokens)[0])
                    end_token_idx = start_token_idx + len(answer_tokens) - 1
                except ValueError:
                    # Print a message and skip if the answer is not found in the context
                    print(f"Answer not found in the context for question: {question_text}")
                    continue  # Skip this answer-context pair

                # Append the processed item to the list
                processed_data.append({
                    'input_ids': encoded_input['input_ids'].squeeze(0),          # Input IDs
                    'attention_mask': encoded_input['attention_mask'].squeeze(0),  # Attention mask
                    'start_positions': torch.tensor([start_token_idx], dtype=torch.long),  # Start index of the answer
                    'end_positions': torch.tensor([end_token_idx], dtype=torch.long)       # End index of the answer
                })

    # Return the list of processed data
    return processed_data

# Example usage: preprocess a dataset named 'bioasq_data'
processed_bioasq_data = preprocess_data(bioasq_data)

## Create a Pytorch Dataset


Using a custom dataset class like BioASQDataset with PyTorch's DataLoader is crucial for structuring data in a machine learning-friendly format, ensuring compatibility with PyTorch's data handling systems, and enabling efficient, scalable, and flexible data processing.

In [ ]:
class BioASQDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = BioASQDataset(processed_bioasq_data)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True)

## Load BioBERT Model

In [ ]:
model = BertForQuestionAnswering.from_pretrained("dmis-lab/biobert-v1.1")

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Train the Model

In [ ]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

#for epoch in range(2):  # Example: 2 training epochs
model.train()
for batch in data_loader:
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    start_positions = batch['start_positions']
    end_positions = batch['end_positions']

    outputs = model(input_ids, attention_mask=attention_mask, start_positions=start_positions, end_positions=end_positions)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print(f"Epoch 1, Loss: {loss.item()}")

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1, Loss: 0.4771046042442322


## Save the model

In [ ]:
save_directory = '/content/drive/My Drive/ECE_7500_NLP'
model_save_path = save_directory + '/bio_bert_qa_model_full.pth'
torch.save(model, model_save_path)

In [ ]:
model = torch.load(model_save_path)
model.eval()  # Set the model to evaluation mode

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

# 1.1 BioBERT Testing (Start from here so you don't have to run the training part)

## Load the trained BioBERT Model

In [ ]:
import torch
from transformers import BertForQuestionAnswering, AutoTokenizer

# Load the model
model_save_path = '/content/drive/My Drive/ECE_7500_NLP/bio_bert_qa_model_full.pth'
biobert_model = torch.load(model_save_path)
biobert_model.eval()  # Set the model to evaluation mode

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

## Find Context

### Create a Context-ID map from the dataset

In [ ]:
def create_context_id_mapping(dataset):
    id_to_context = {}

    for item in dataset:
        context_id = item['id']
        snippets = item.get('snippets', [])
        context_texts = [snippet['text'] for snippet in snippets]
        id_to_context[context_id] = ' '.join(context_texts)

    return id_to_context

# Assuming bioasq_data is your dataset
id_to_context_mapping = create_context_id_mapping(bioasq_data)

### Create a Semantic Search System

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained sentence transformer model
semantic_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

# Precompute embeddings for the questions in your dataset
question_embeddings = {}
for item in bioasq_data:
    question_text = item['body']
    question_embeddings[item['id']] = semantic_transformer_model.encode(question_text, convert_to_tensor=True)

.gitattributes:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

train_script.py:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

### Retrieving Context for User Queries

In [ ]:
def get_similar_context(query, question_embeddings, id_to_context_mapping, top_k=5):
    query_embedding = semantic_transformer_model.encode(query, convert_to_tensor=True)
    similarities = []

    for id, q_embedding in question_embeddings.items():
        similarity = util.pytorch_cos_sim(query_embedding, q_embedding)
        similarities.append((id, similarity.item()))

    # Sort by similarity
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Get top_k similar ids
    similar_ids = [sim[0] for sim in similarities[:top_k]]
    return [id_to_context_mapping[id] for id in similar_ids if id in id_to_context_mapping]

### Answering the User's Question

In [ ]:
# Example usage
user_query = "Are long non coding RNAs spliced?"
biobert_similar_contexts = get_similar_context(user_query, question_embeddings, id_to_context_mapping)

In [ ]:
user_query

'Are long non coding RNAs spliced?'

In [ ]:
biobert_similar_contexts

['Our analyses indicate that lncRNAs are generated through pathways similar to that of protein-coding genes, with similar histone-modification profiles, splicing signals, and exon/intron lengths. For alternative exons and long noncoding RNAs, splicing tends to occur later, and the latter might remain unspliced in some cases. bosome-mapping data to identify lncRNAs of Caenorhabditis elegans. We found 170 long intervening ncRNAs (lincRNAs), which had single- or multiexonic structures that did not overlap protein-coding transcripts, and about sixty antisense lncRNAs (ancRNAs), which were complementary to protein-coding transcripts We introduce an approach to predict spliced lncRNAs in vertebrate genomes combining comparative genomics and machine learning. Owing to similar alternative splicing pattern to mRNAs, the concept of lncRNA genes was put forward to help systematic understanding of lncRNAs.  Our synthesis of recent studies suggests that neither size, presence of a poly-A tail, spli

In [ ]:
def answer_question(question, context, model, tokenizer):
    # Tokenize input
    inputs = tokenizer.encode_plus(question, context, add_special_tokens=True, return_tensors='pt', max_length=512, truncation=True)

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the start and end positions of the answer in the context
    answer_start_scores, answer_end_scores = outputs.start_logits, outputs.end_logits
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    # Convert tokens to the answer string
    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][answer_start:answer_end]))

    return answer

In [ ]:
def biobert_answer_user_question(user_query, similar_contexts, biobert_model, biobert_tokenizer):
    for context in similar_contexts:
        answer = answer_question(user_query, context, biobert_model, biobert_tokenizer)
        if answer:
            return answer
    return "Sorry, I couldn't find an answer."

In [ ]:
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
biobert_answer_user_question(user_query, biobert_similar_contexts, biobert_model, biobert_tokenizer)

'spliced'

In [ ]:
user_query = "what is RANKL?"
similar_contexts = get_similar_context(user_query, question_embeddings, id_to_context_mapping)
#sample_context = "Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes"

biobert_answer = biobert_answer_user_question(user_query, biobert_similar_contexts, biobert_model, biobert_tokenizer)
print("Answer:", biobert_answer)

Answer: lncRNAs are generated through


In [ ]:
similar_contexts

['Osteoprotegerin (OPG) is a soluble secreted factor that acts as a decoy receptor for receptor activator of NF-κB ligand (RANKL)  Osteoprotegerin (OPG) is a secreted glycoprotein and a member of the tumor necrosis factor receptor superfamily. It usually functions in bone remodeling, by inhibiting osteoclastogenesis through interaction with a receptor activator of the nuclear factor κB (RANKL). e RANKL/OPG ratio secreted by osteoblasts increased and RANK expression by osteoclasts increased, leading to increased osteoclastogenesis Osteoprotegerin (OPG) is an essential secreted protein in bone turnover due to its role as a decoy receptor for the Receptor Activator of Nuclear Factor-kB ligand (RANKL) in the osteoclasts, thus inhibiting their differentiation We identify a TNFSF11 transcript variant that extends the originally identified transcript encoding secreted RANKL. Activated human T cells express alternative mRNA transcripts encoding a secreted form of RANKL. OPG, on the other hand,

# 1.2 Set Up Wikipedia Fallback

In [ ]:
!pip install requests

import requests

In [ ]:
def search_wikipedia(query):
    """Search Wikipedia for a page and return the page title."""
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    response = requests.get(search_url, params=params)
    results = response.json()['query']['search']
    if results:
        return results[0]['title']  # Return the title of the first result
    return None

def fetch_wikipedia_context(query):
    """Fetch the context from Wikipedia based on a query."""
    WIKI_API_URL = "https://en.wikipedia.org/w/api.php"

    # First, search Wikipedia to get the page title
    page_title = search_wikipedia(query)
    if not page_title:
        return "No relevant information found on Wikipedia."

    params = {
        "action": "query",
        "format": "json",
        "titles": page_title,
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
    }

    response = requests.get(WIKI_API_URL, params=params)
    data = response.json()

    page = next(iter(data['query']['pages'].values()))
    if "extract" in page:
        return page['extract']
    else:
        return "No detailed information available."

In [ ]:
# Test the function
wiki_answer = fetch_wikipedia_context("what is Hirschsprung disease")
print(wiki_answer)

Hirschsprung's disease (HD or HSCR) is a birth defect in which nerves are missing from parts of the intestine. The most prominent symptom is constipation. Other symptoms may include vomiting, abdominal pain, diarrhea and slow growth. Most children develop signs and symptoms shortly after birth. However, others may be diagnosed later in infancy or early childhood. About half of all children with Hirschsprung's disease are diagnosed in the first year of life. Complications may include enterocolitis, megacolon, bowel obstruction and intestinal perforation.The disorder may occur by itself or in association with other genetic disorders such as Down syndrome or Waardenburg syndrome. About half of isolated cases are linked to a specific genetic mutation, and about 20% occur within families. Some of these occur in an autosomal dominant manner. The cause of the remaining cases is unclear. If otherwise normal parents have one child with the condition, the next child has a 4% risk of being affect

In [ ]:
# Test the function
wiki_answer = fetch_wikipedia_context("Is RANKL secreted from the cells?")
print(wiki_answer)

Receptor activator of nuclear factor kappa-Β ligand (RANKL), also known as tumor necrosis factor ligand superfamily member 11 (TNFSF11), TNF-related activation-induced cytokine (TRANCE), osteoprotegerin ligand (OPGL), and osteoclast differentiation factor (ODF), is a protein that in humans is encoded by the TNFSF11 gene.RANKL is known as a type II membrane protein and is a member of the tumor necrosis factor (TNF) superfamily.  RANKL has been identified to affect the immune system and control bone regeneration and remodeling. RANKL is an apoptosis regulator gene, a binding partner of osteoprotegerin (OPG), a ligand for the receptor RANK and controls cell proliferation by modifying protein levels of Id4, Id2 and cyclin D1. RANKL is expressed in several tissues and organs including: skeletal muscle, thymus, liver, colon, small intestine, adrenal gland, osteoblast, mammary gland epithelial cells, prostate and pancreas. Variation in concentration levels of RANKL throughout several organs r

# samples example

In [ ]:
sample_question = "what is Hirschsprung disease?" #a mendelian or a multifactorial disorder
sample_context = "Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes"

# 2. Train a BERT Model on SQuAD for Semantic Understanding

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install datasets
!pip install huggingface-hub

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-oh4w1xmt
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-oh4w1xmt
  Resolved https://github.com/huggingface/transformers.git to commit 2c658b5a4282f2e824b4e23dc3bcda7ef27d5827
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.36.0.dev0-py3-none-any.whl size=8119970 sha256=d5937e7e3794b19fa8ce25803e8c8c5ede8a823d11d16a14ea6f3d7b1772fc6c
  Stored in directory: /tmp/pip-ephem-wheel-cache-koclqgom/wheels/e7/9c/5b/e1a9c8007c343041e61cc484433d512ea9274272e3fcbe7c16
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.35.2
    Uninstalling transformers-4.35.2:
      Successfully uninstalled transformers-4.35.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

datasets = load_dataset("squad")
print(datasets["train"][0])

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


## Preprocess the SQUAD dataset using Transformers Tokenizer

In [ ]:
from transformers import AutoTokenizer

distilbert_model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(distilbert_model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
max_length = 484  # The maximum length of a feature (question and context)
doc_stride = (
    150  # The authorized overlap between two part of the context when splitting
)
# it is needed.

In [ ]:
def prepare_train_features(examples):
    # Tokenize our examples with truncation and padding, but keep the overflows using a
    # stride. This results in one example possible giving several features when a context is long,
    # each of those features having a context that overlaps a bit the context of the previous
    # feature.
    examples["question"] = [q.lstrip() for q in examples["question"]]
    examples["context"] = [c.lstrip() for c in examples["context"]]
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Since one example might give us several features if it has a long context, we need a
    # map from a feature to its corresponding example. This key gives us just that.
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    # The offset mappings will give us a map from token to character position in the original
    # context. This will help us compute the start_positions and end_positions.
    offset_mapping = tokenized_examples.pop("offset_mapping")

    # Let's label those examples!
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        # We will label impossible answers with the index of the CLS token.
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Grab the sequence corresponding to that example (to know what is the context and what
        # is the question).
        sequence_ids = tokenized_examples.sequence_ids(i)

        # One example can give several spans, this is the index of the example containing this
        # span of text.
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        # If no answers are given, set the cls_index as answer.
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Start/end character index of the answer in the text.
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Start token index of the current span in the text.
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            # End token index of the current span in the text.
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # Detect if the answer is out of the span (in which case this feature is labeled with the
            # CLS index).
            if not (
                offsets[token_start_index][0] <= start_char
                and offsets[token_end_index][1] >= end_char
            ):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # Otherwise move the token_start_index and token_end_index to the two ends of the
                # answer.
                # Note: we could go after the last offset if the answer is the last word (edge
                # case).
                while (
                    token_start_index < len(offsets)
                    and offsets[token_start_index][0] <= start_char
                ):
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

In [ ]:
tokenized_datasets = datasets.map(
    prepare_train_features,
    batched=True,
    remove_columns=datasets["train"].column_names,
    num_proc=3,
)

Map (num_proc=3):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=3):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
train_set = tokenized_datasets["train"].with_format("numpy")[
    :
]  # Load the whole dataset as a dict of numpy arrays
validation_set = tokenized_datasets["validation"].with_format("numpy")[:]

## Fine-Tuning BERT Model

In [ ]:
from transformers import TFAutoModelForQuestionAnswering

distilbert_model = TFAutoModelForQuestionAnswering.from_pretrained(distilbert_model_checkpoint)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForQuestionAnswering: ['vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForQuestionAnswering from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForQuestionAnswering from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForQuestionAnswering were not initialized from the PyTorch model and are newly initialized: ['qa_outputs.weight', 'qa_outputs.bias']
You should probably TRAIN this model on a down-stream task to be able to use it

In [ ]:
import tensorflow as tf
from tensorflow import keras

optimizer = keras.optimizers.Adam(learning_rate=5e-5)

In [ ]:
# Optionally uncomment the next line for float16 training
keras.mixed_precision.set_global_policy("mixed_float16")

distilbert_model.compile(optimizer=optimizer)

In [ ]:
distilbert_model.fit(train_set, validation_data=validation_set, epochs=2)

Epoch 1/2
2745/2745 [==============================] - 1550s 557ms/step - loss: 1.5323 - val_loss: 1.1645
Epoch 2/2
2745/2745 [==============================] - 1527s 556ms/step - loss: 0.9305 - val_loss: 1.1770


In [ ]:
# Tokenize the input text pair and convert to tensors
inputs = tokenizer(sample_context, sample_question, return_tensors="np")

# Get model outputs (start and end logits)
outputs = distilbert_model(inputs)

# Extract start and end positions
start_position = tf.argmax(outputs.start_logits, axis=1)
end_position = tf.argmax(outputs.end_logits, axis=1)
print("Start and End Position: ",int(start_position), int(end_position[0]))

# Get the answer tokens
answer_tokens = inputs["input_ids"][0, int(start_position) : int(end_position) + 1]

# Decode the tokens to a string
answer = tokenizer.decode(answer_tokens)
print(answer)

Start and End Position:  12 54
a multifactorial, non - mendelian disorder in which rare high - penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes


## Save DistilBert Model

In [ ]:
model_save_path = "/content/drive/My Drive/ECE_7500_NLP/distilbert_model"
distilbert_model.save(model_save_path)

In [ ]:
model_save_path = "/content/drive/My Drive/ECE_7500_NLP/distilbert_model2"
distilbert_model.save_pretrained(model_save_path)

# 2.1 Load the Trained DistilBert Model

In [ ]:
from transformers import TFDistilBertForQuestionAnswering
import tensorflow as tf

model_load_path = "/content/drive/My Drive/ECE_7500_NLP/distilbert_model2"
loaded_distilbert_model = TFDistilBertForQuestionAnswering.from_pretrained(model_load_path)

Some layers from the model checkpoint at /content/drive/My Drive/ECE_7500_NLP/distilbert_model2 were not used when initializing TFDistilBertForQuestionAnswering: ['dropout_19']
- This IS expected if you are initializing TFDistilBertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForQuestionAnswering were not initialized from the model checkpoint at /content/drive/My Drive/ECE_7500_NLP/distilbert_model2 and are newly initialized: ['dropout_39']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Tokenize and convert to tensors
encoded_input = tokenizer(sample_context, sample_question, return_tensors="np")

# Get model predictions
outputs = loaded_distilbert_model(encoded_input)

# Extract start and end logits
start_position = tf.argmax(outputs.start_logits, axis=1)
end_position = tf.argmax(outputs.end_logits, axis=1)

# Extract the answer tokens and decode
answer_tokens = encoded_input["input_ids"][0, int(start_position) : int(end_position) + 1]
answer = tokenizer.decode(answer_tokens)

print("Answer:", answer)

Answer: a multifactorial, non - mendelian disorder in which rare high - penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

def dummy_loss(y_true, y_pred):
    # Exact implementation of the custom loss function
    # ...
    return tf.constant(0.0)

In [ ]:
model_load_path = "/content/drive/My Drive/ECE_7500_NLP/distilbert_model"
loaded_distilbert_model = load_model(model_load_path, custom_objects={'dummy_loss': dummy_loss})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

distilbert_model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(distilbert_model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
encoded_input = tokenizer.encode_plus(sample_context, sample_question, add_special_tokens=True, return_tensors="tf")

# Assuming loaded_distilbert_model is your loaded model
predictions = loaded_distilbert_model(encoded_input)

# Extract start and end logits
start_logits = predictions['start_logits']
end_logits = predictions['end_logits']

start_position = tf.argmax(start_logits, axis=1).numpy()[0]
end_position = tf.argmax(end_logits, axis=1).numpy()[0] + 1

# Extract the answer tokens and decode
answer_tokens = encoded_input["input_ids"][0, start_position:end_position]
answer = tokenizer.decode(answer_tokens)

print("Answer:", answer) #Answer: a multifactorial, non - mendelian disorder in which rare high - penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes

Answer: a multifactorial, non - mendelian disorder in which rare high - penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes


# 2.1 distilbert-base-cased-distilled-squad

## Test DistilBert Model

In [ ]:
from transformers import DistilBertTokenizer

# Load the tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Example question and context
question = "What is the capital of France?"
context = "France is a country in Europe. The capital of France is Paris."

sample_question = "is Hirschsprung disease a mendelian or a multifactorial disorder?"
sample_context = "Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes"

In [ ]:
from transformers import pipeline

# Load the question-answering pipeline with the specified model
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
answer = qa_pipeline(question=sample_question, context=sample_context)

In [ ]:
print(answer['answer'])

multifactorial, non-mendelian disorder


# 3. Read and Process PDF

In [ ]:
!pip install langchain sentence-transformers
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.7 MB/s eta 0:00:00


In [ ]:
#get PDF Texts
import PyPDF2

def extract_text_from_pdf(file_path, start_page, end_page):
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)

        # Adjusting for zero-based indexing
        start_page -= 1
        end_page -= 1

        # Extracting text from each page
        text = ''
        count = 0
        for page_num in range(start_page, end_page + 1):
            try:
                count += 1
                page = reader.pages[page_num]
                text += page.extract_text() + "\n"
            except IndexError:
                print(f"Page {page_num + 1} is out of range.")
                break

        return text, count

# Example usage
file_path = '/content/drive/My Drive/ECE_7500_NLP/Current_Essentials_of_Medicine.pdf'
extracted_text, count = extract_text_from_pdf(file_path, 30, 90)
print(count)

61


In [ ]:
extracted_text[0]

'C'

In [ ]:
extracted_text

'Chapter 1 Cardiovascular Diseases 17\n1 Cor Pulmonale\n■Essentials of Diagnosis\n•Heart failure resulting from pulmonary disease\n•Most commonly due to COPD; other causes include pulmonaryﬁbrosis, pneumoconioses, recurrent pulmonary emboli, primarypulmonary hypertension, sleep apnea, and kyphoscoliosis\n•Clinical manifestations are due to both the underlying pulmonarydisease and the right ventricular failure\n•Chest x-ray reveals an enlarged right ventricle and pulmonaryartery; electrocardiography may show right axis deviation, rightventricular hypertrophy, and tall, peaked P waves (P pulmonale)in the face of low QRS voltage\n•Pulmonary function tests usually conﬁrm the presence of underlyinglung disease, and echocardiography will show right ventriculardilation but normal left ventricular function and elevated rightventricular systolic pressures\n■Differential Diagnosis\nOther causes of right ventricular failure:\n•Left ventricular failure (due to any cause)\n•Pulmonary stenosis\n•Lef

In [ ]:
def process_extracted_text(text):
    words = []
    current_word = ''

    for char in text:
        if char.isalpha():  # Check if character is a letter
            current_word += char  # Add to current word
        elif current_word:
            # If there's a current word and we hit a non-letter, we assume the word ended
            words.append(current_word)
            current_word = ''

    # Add the last word if there is one
    if current_word:
        words.append(current_word)

    return words

# Assuming extracted_text is a string with the extracted text
words = process_extracted_text(extracted_text)
print(words)

['Chapter', 'Cardiovascular', 'Diseases', 'Cor', 'Pulmonale', 'Essentials', 'of', 'Diagnosis', 'Heart', 'failure', 'resulting', 'from', 'pulmonary', 'disease', 'Most', 'commonly', 'due', 'to', 'COPD', 'other', 'causes', 'include', 'pulmonaryﬁbrosis', 'pneumoconioses', 'recurrent', 'pulmonary', 'emboli', 'primarypulmonary', 'hypertension', 'sleep', 'apnea', 'and', 'kyphoscoliosis', 'Clinical', 'manifestations', 'are', 'due', 'to', 'both', 'the', 'underlying', 'pulmonarydisease', 'and', 'the', 'right', 'ventricular', 'failure', 'Chest', 'x', 'ray', 'reveals', 'an', 'enlarged', 'right', 'ventricle', 'and', 'pulmonaryartery', 'electrocardiography', 'may', 'show', 'right', 'axis', 'deviation', 'rightventricular', 'hypertrophy', 'and', 'tall', 'peaked', 'P', 'waves', 'P', 'pulmonale', 'in', 'the', 'face', 'of', 'low', 'QRS', 'voltage', 'Pulmonary', 'function', 'tests', 'usually', 'conﬁrm', 'the', 'presence', 'of', 'underlyinglung', 'disease', 'and', 'echocardiography', 'will', 'show', 'right

In [ ]:
len(words)

12887

In [ ]:
paragraph = ' '.join(words)
print(paragraph)

Chapter Cardiovascular Diseases Cor Pulmonale Essentials of Diagnosis Heart failure resulting from pulmonary disease Most commonly due to COPD other causes include pulmonaryﬁbrosis pneumoconioses recurrent pulmonary emboli primarypulmonary hypertension sleep apnea and kyphoscoliosis Clinical manifestations are due to both the underlying pulmonarydisease and the right ventricular failure Chest x ray reveals an enlarged right ventricle and pulmonaryartery electrocardiography may show right axis deviation rightventricular hypertrophy and tall peaked P waves P pulmonale in the face of low QRS voltage Pulmonary function tests usually conﬁrm the presence of underlyinglung disease and echocardiography will show right ventriculardilation but normal left ventricular function and elevated rightventricular systolic pressures Differential Diagnosis Other causes of right ventricular failure Left ventricular failure due to any cause Pulmonary stenosis Left to right shunt causing Eisenmenger s syndro

In [ ]:
len(paragraph.split())

12887

In [ ]:
!pip install langchain

In [ ]:
def chunk_text(text, max_chunk_words):
    """
    Chunk text into segments with a maximum number of words.

    Args:
    text (str): The text to be chunked.
    max_chunk_words (int): Maximum number of words per chunk.

    Returns:
    list: A list of text chunks.
    """
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(current_chunk) >= max_chunk_words:
            chunks.append(' '.join(current_chunk))
            current_chunk = []

    # Add the last chunk if there are remaining words
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Example usage
max_chunk_words = 100  # Adjust this to change the chunk size in terms of word count
chunks = chunk_text(paragraph, max_chunk_words)

# Displaying the first chunk as an example
print(chunks[0])

Chapter Cardiovascular Diseases Cor Pulmonale Essentials of Diagnosis Heart failure resulting from pulmonary disease Most commonly due to COPD other causes include pulmonaryﬁbrosis pneumoconioses recurrent pulmonary emboli primarypulmonary hypertension sleep apnea and kyphoscoliosis Clinical manifestations are due to both the underlying pulmonarydisease and the right ventricular failure Chest x ray reveals an enlarged right ventricle and pulmonaryartery electrocardiography may show right axis deviation rightventricular hypertrophy and tall peaked P waves P pulmonale in the face of low QRS voltage Pulmonary function tests usually conﬁrm the presence of underlyinglung disease and echocardiography will show right ventriculardilation but normal left ventricular function


In [ ]:
len(chunks)

129

In [ ]:
def answer_question_with_pipeline(question, chunks, confidence_threshold=0.5):
    for chunk in chunks:
        # Use the QA pipeline to find an answer in the current chunk
        result = qa_pipeline(question=question, context=chunk)

        # Check if the answer is confident enough
        if result['score'] >= confidence_threshold:
            #print(f"{result['answer']}")
            return result['answer']
            break
    else:
        # This else block executes if no break was hit in the loop, meaning no answer was found
        print("Answer not found in the provided context.")

# Example usage
question = "What is Cardiovascular Diseases?"
answer_question_with_pipeline(question, chunks)


'Hypertrophic Obstructive Cardiomyopathy'

# 4. ChatBot

### BioBERT (Model)

In [ ]:
from transformers import BertForQuestionAnswering, AutoTokenizer

def biobert_finetuned (sample_question):
  biobert_similar_contexts = get_similar_context(sample_question, question_embeddings, id_to_context_mapping)
  biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
  biober_answer = biobert_answer_user_question(sample_question, biobert_similar_contexts, biobert_model, biobert_tokenizer)
  return biober_answer

### WikiPedia (Context)

In [ ]:
def wiki_context(sample_question):
  return fetch_wikipedia_context(sample_question)

### DistilBERT (Model)

In [ ]:
from transformers import TFDistilBertForQuestionAnswering
from transformers import AutoTokenizer
import tensorflow as tf

model_load_path = "/content/drive/My Drive/ECE_7500_NLP/distilbert_model2"
loaded_distilbert_model = TFDistilBertForQuestionAnswering.from_pretrained(model_load_path)

distilbert_model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(distilbert_model_checkpoint)

Some layers from the model checkpoint at /content/drive/My Drive/ECE_7500_NLP/distilbert_model2 were not used when initializing TFDistilBertForQuestionAnswering: ['dropout_19']
- This IS expected if you are initializing TFDistilBertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForQuestionAnswering were not initialized from the model checkpoint at /content/drive/My Drive/ECE_7500_NLP/distilbert_model2 and are newly initialized: ['dropout_97']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def distilbert_finetuned(sample_question, sample_context):
    # Tokenize and convert to tensors
    encoded_input = tokenizer(sample_context, sample_question, return_tensors="np")

    # Get model predictions
    outputs = loaded_distilbert_model(encoded_input)

    # Extract start and end logits
    start_position = tf.argmax(outputs.start_logits, axis=1)
    end_position = tf.argmax(outputs.end_logits, axis=1)

    # Extract the answer tokens and decode
    answer_tokens = encoded_input["input_ids"][0, int(start_position) : int(end_position) + 1]
    answer = tokenizer.decode(answer_tokens)

    return answer

In [ ]:
sample_question = "what is cardiovascular disease?"
sample_context = "Cardiovascular disease (CVD) is any disease involving the heart or blood vessels. CVDs constitute a class of diseases that includes: coronary artery diseases (e.g. angina, heart attack), heart failure, hypertensive heart disease, rheumatic heart disease, cardiomyopathy, arrhythmia, congenital heart disease, valvular heart disease, carditis, aortic aneurysms, peripheral artery disease, thromboembolic disease, and venous thrombosis.The underlying mechanisms vary depending on the disease. It is estimated that dietary risk factors are associated with 53% of CVD deaths. Coronary artery disease, stroke, and peripheral artery disease involve atherosclerosis. This may be caused by high blood pressure, smoking, diabetes mellitus, lack of exercise, obesity, high blood cholesterol, poor diet, excessive alcohol consumption,  and poor sleep, among other things. High blood pressure is estimated to account for approximately 13% of CVD deaths, while tobacco accounts for 9%, diabetes 6%, lack of exercise 6%, and obesity 5%. Rheumatic heart disease may follow untreated strep throat.It is estimated that up to 90% of CVD may be preventable. Prevention of CVD involves improving risk factors through: healthy eating, exercise, avoidance of tobacco smoke and limiting alcohol intake. Treating risk factors, such as high blood pressure, blood lipids and diabetes is also beneficial. Treating people who have strep throat with antibiotics can decrease the risk of rheumatic heart disease. The use of aspirin in people who are otherwise healthy is of unclear benefit.Cardiovascular diseases are the leading cause of death worldwide except Africa. Together CVD resulted in 17.9 million deaths (32.1%) in 2015, up from 12.3 million (25.8%) in 1990. Deaths, at a given age, from CVD are more common and have been increasing in much of the developing world, while rates have declined in most of the developed world since the 1970s. Coronary artery disease and stroke account for 80% of CVD deaths in males and 75% of CVD deaths in females. Most cardiovascular disease affects older adults. In the United States 11% of people between 20 and 40 have CVD, while 37% between 40 and 60, 71% of people between 60 and 80, and 85% of people over 80 have CVD. The average age of death from coronary artery disease in the developed world is around 80, while it is around 68 in the developing world. CVD is typically diagnosed seven to ten years earlier in men than in women.: 48"
distilbert_finetuned(sample_question, sample_context)

Answer: coronary artery diseases ( e. g. angina, heart attack ), heart failure, hypertensive heart disease, rheumatic heart disease, cardiomyopathy, arrhythmia, congenital heart disease, valvular heart disease, carditis, aortic aneurysms, peripheral artery disease, thromboembolic disease, and venous thrombosis. The underlying mechanisms vary depending on the disease. It is estimated that dietary risk factors are associated with 53 % of CVD deaths. Coronary artery disease, stroke, and peripheral artery disease involve atherosclerosis. This may be caused by high blood pressure, smoking, diabetes mellitus, lack of exercise, obesity, high blood cholesterol, poor diet, excessive alcohol consumption, and poor sleep, among other things. High blood pressure is estimated to account for approximately 13 % of CVD deaths, while tobacco accounts for 9 %, diabetes 6 %, lack of exercise 6 %, and obesity 5 %. Rheumatic heart disease may follow untreated strep throat. It is estimated that up to 90 % of

'coronary artery diseases ( e. g. angina, heart attack ), heart failure, hypertensive heart disease, rheumatic heart disease, cardiomyopathy, arrhythmia, congenital heart disease, valvular heart disease, carditis, aortic aneurysms, peripheral artery disease, thromboembolic disease, and venous thrombosis. The underlying mechanisms vary depending on the disease. It is estimated that dietary risk factors are associated with 53 % of CVD deaths. Coronary artery disease, stroke, and peripheral artery disease involve atherosclerosis. This may be caused by high blood pressure, smoking, diabetes mellitus, lack of exercise, obesity, high blood cholesterol, poor diet, excessive alcohol consumption, and poor sleep, among other things. High blood pressure is estimated to account for approximately 13 % of CVD deaths, while tobacco accounts for 9 %, diabetes 6 %, lack of exercise 6 %, and obesity 5 %. Rheumatic heart disease may follow untreated strep throat. It is estimated that up to 90 % of CVD ma

In [ ]:
from transformers import DistilBertTokenizer
from transformers import pipeline

def distilber_pretrained(sample_question, sample_context):
  # Load the tokenizer
  pretained_distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
  # Load the question-answering pipeline with the specified model
  qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
  answer = qa_pipeline(question=sample_question, context=sample_context)

  return answer['answer']

### PDF Reader (Context)

In [ ]:
def pdf_answer(sample_question):
  return answer_question_with_pipeline(sample_question, chunks)

In [ ]:
pdf_answer("What is Cardiovascular Diseases?")

'Hypertrophic Obstructive Cardiomyopathy'

## Combine

In [ ]:
def chatbot():
    print("Welcome to the medical chatbot. Enter a query or Type 'exit' to end the conversation.")

    while True:
        # Get user input
        user_input = input("You: ")

        # Check if the user wants to exit
        if user_input.lower() == 'exit':
            print("Exiting the chatbot. Goodbye!")
            break

        biobert_finetuned_answer = biobert_finetuned(user_input)
        wiki_context_answer = wiki_context(user_input)
        from_pdf_answer = pdf_answer(user_input)
        distilbert_finetuned_answer = distilbert_finetuned(user_input, wiki_context_answer)
        distilber_pretrained_answer = distilber_pretrained(user_input, wiki_context_answer)

        # Print the chatbot's response
        print(f"Chatbot: \nbiobert_finetuned_answer- {biobert_finetuned_answer}, \n\nwiki_context_answer- {wiki_context_answer}, \n\npdf_answer - {from_pdf_answer}, \n\ndistilbert_finetuned_answer - {distilbert_finetuned_answer}, \n\ndistilber_pretrained_answer = {distilber_pretrained_answer}")


# Run the chatbot
chatbot()

Welcome to the medical chatbot. Enter a query or Type 'exit' to end the conversation.
You: what is cardiovascular disease
Answer: coronary artery diseases ( e. g. angina, heart attack ), heart failure, hypertensive heart disease, rheumatic heart disease, cardiomyopathy, arrhythmia, congenital heart disease, valvular heart disease, carditis, aortic aneurysms, peripheral artery disease, thromboembolic disease, and venous thrombosis. The underlying mechanisms vary depending on the disease. It is estimated that dietary risk factors are associated with 53 % of CVD deaths. Coronary artery disease, stroke, and peripheral artery disease involve atherosclerosis. This may be caused by high blood pressure, smoking, diabetes mellitus, lack of exercise, obesity, high blood cholesterol, poor diet, excessive alcohol consumption, and poor sleep, among other things. High blood pressure is estimated to account for approximately 13 % of CVD deaths, while tobacco accounts for 9 %, diabetes 6 %, lack of ex